In [ ]:
#

In [ ]:
teste()

Combinações únicas:
[(1, 3, 2, 4), (2, 1, 3, 4), (2, 1, 4, 3), (3, 2, 1, 4), (3, 2, 4, 1), (4, 1, 3, 2), (4, 2, 1, 3), (4, 3, 2, 1)]

Quantidade de combinações únicas:
8

Frequência de cada combinação:
PreFormAnswers
(4, 2, 1, 3)    3
(3, 2, 4, 1)    3
(1, 3, 2, 4)    3
(3, 2, 1, 4)    2
(4, 3, 2, 1)    1
(4, 1, 3, 2)    1
(2, 1, 3, 4)    1
(2, 1, 4, 3)    1
Name: count, dtype: int64

Frequência no dungeon_frame final:
PreFormAnswers
[1, 3, 2, 4]    26
[4, 1, 3, 2]    25
[2, 1, 3, 4]    24
[4, 2, 1, 3]    18
[3, 2, 1, 4]    11
[3, 2, 4, 1]     9
[2, 1, 4, 3]     5
[4, 3, 2, 1]     1
Name: count, dtype: int64

Total de colunas: 57

  1. CompletedAchievementQuests
  2. CompletedCreativityQuests
  3. CompletedDamageQuests
  4. CompletedExchangeQuests
  5. CompletedExploreQuests
  6. CompletedGatherQuests
  7. CompletedGiveQuests
  8. CompletedGoToQuests
  9. CompletedImmersionQuests
 10. CompletedKillQuests
 11. CompletedListenQuests
 12. CompletedMasteryQuests
 13. CompletedQuests
 14. C

In [ ]:
def hit_rate_at_k(pred_df, test_df, k=5):
    hits = 0
    total = 0

    for u in test_df['User'].unique():

        # item real
        true_item = test_df[test_df['User'] == u]['Item'].iloc[0]

        # top-k recomendações
        user_preds = pred_df[pred_df['User'] == u]
        topk = user_preds.sort_values('Prediction', ascending=False).head(k)
        recomendados = set(topk['Item'])

        if true_item in recomendados:
            hits += 1

        total += 1

    return hits / total if total > 0 else 0

def mrr(pred_df, test_df):
    rr = []

    for u in test_df['User'].unique():

        true_item = test_df[test_df['User'] == u]['Item'].iloc[0]

        user_preds = pred_df[pred_df['User'] == u]
        ranked = user_preds.sort_values('Prediction', ascending=False)['Item'].tolist()

        if true_item in ranked:
            rank = ranked.index(true_item) + 1
            rr.append(1 / rank)
        else:
            rr.append(0)

    return np.mean(rr)

In [ ]:

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

# =========================================================
# LEAVE ONE OUT
# =========================================================

def leave_one_out_split(df, user_col='User'):

    train_list = []
    test_list = []

    for user, group in df.groupby(user_col):

        # precisa ter pelo menos 2 interações
        if len(group) < 2:
            continue

        # remove 1 item relevante
        test_row = group.sample(1, random_state=42)

        # restante vai pro treino
        train_rows = group.drop(test_row.index)

        train_list.append(train_rows)
        test_list.append(test_row)

    train_df = pd.concat(train_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)

    return train_df, test_df


# =========================================================
# HIT RATE @ K
# =========================================================

def hit_rate_at_k(rec_df, test_df, k=10):

    hits = 0
    total = 0

    for user in test_df["User"].unique():

        relevant_items = set(
            test_df[test_df["User"] == user]["Item"]
        )

        top_k = (
            rec_df[rec_df["User"] == user]
            .sort_values("Score", ascending=False)
            .head(k)
        )

        recommended_items = set(top_k["Item"])

        if len(relevant_items & recommended_items) > 0:
            hits += 1

        total += 1

    return hits / total if total > 0 else 0


# =========================================================
# MRR
# =========================================================

def mrr(rec_df, test_df):

    reciprocal_ranks = []

    for user in test_df["User"].unique():

        relevant_items = set(
            test_df[test_df["User"] == user]["Item"]
        )

        ranked_items = (
            rec_df[rec_df["User"] == user]
            .sort_values("Score", ascending=False)
        )["Item"].tolist()

        found = False

        for rank, item in enumerate(ranked_items, start=1):

            if item in relevant_items:
                reciprocal_ranks.append(1 / rank)
                found = True
                break

        if not found:
            reciprocal_ranks.append(0)

    return np.mean(reciprocal_ranks)


# =========================================================
# PRECISION @ K
# =========================================================

def precision_at_k(rec_df, test_df, k=10):

    precisions = []

    for user in test_df["User"].unique():

        relevant_items = set(
            test_df[test_df["User"] == user]["Item"]
        )

        top_k = (
            rec_df[rec_df["User"] == user]
            .sort_values("Score", ascending=False)
            .head(k)
        )

        recommended_items = set(top_k["Item"])

        hits = len(recommended_items & relevant_items)

        precisions.append(hits / k)

    return np.mean(precisions)


# =========================================================
# RECALL @ K
# =========================================================

def recall_at_k(rec_df, test_df, k=10):

    recalls = []

    for user in test_df["User"].unique():

        relevant_items = set(
            test_df[test_df["User"] == user]["Item"]
        )

        top_k = (
            rec_df[rec_df["User"] == user]
            .sort_values("Score", ascending=False)
            .head(k)
        )

        recommended_items = set(top_k["Item"])

        hits = len(recommended_items & relevant_items)

        recalls.append(
            hits / len(relevant_items)
        )

    return np.mean(recalls)


# =========================================================
# NDCG
# =========================================================

def ndcg_at_k(rec_df, test_df, k=10):

    ndcg_scores = []

    for user in test_df["User"].unique():

        top_k = (
            rec_df[rec_df["User"] == user]
            .sort_values("Score", ascending=False)
            .head(k)
        )

        recommended_items = top_k["Item"].tolist()

        relevant_items = set(
            test_df[test_df["User"] == user]["Item"]
        )

        y_true = [
            1 if item in relevant_items else 0
            for item in recommended_items
        ]

        y_score = top_k["Score"].tolist()

        if np.sum(y_true) > 0:
            score = ndcg_score([y_true], [y_score])
            ndcg_scores.append(score)
        else:
            ndcg_scores.append(0)

    return np.mean(ndcg_scores)


# =========================================================
# GERAR RANKING
# =========================================================

def generate_recommendations(
        train_matrix,
        similarity_matrix,
        method="user"
):

    recommendations = []

    for user in train_matrix.index:

        for item in train_matrix.columns:

            # prever apenas itens não vistos
            if train_matrix.loc[user, item] == 0:

                try:

                    if method == "user":

                        score = predict_user_based(
                            user,
                            item,
                            train_matrix,
                            similarity_matrix
                        )

                    else:

                        score = predict_item_based(
                            user,
                            item,
                            train_matrix,
                            similarity_matrix
                        )

                    if pd.isna(score):
                        score = 0

                    recommendations.append({
                        "User": user,
                        "Item": item,
                        "Score": score
                    })

                except:
                    pass

    rec_df = pd.DataFrame(recommendations)

    if len(rec_df) == 0:
        return pd.DataFrame(
            columns=["User", "Item", "Score"]
        )

    rec_df = rec_df.sort_values(
        ["User", "Score"],
        ascending=[True, False]
    )

    return rec_df


# =========================================================
# AVALIAÇÃO PRINCIPAL
# =========================================================

def evaluate_collaborative_filtering(
        full_df,
        method,
        top_k=10
):

    print("\n======================================")
    print("LEAVE ONE OUT GLOBAL")
    print("======================================")

    full_df = full_df.copy()

    # ----------------------------------
    # converter listas em string
    # ----------------------------------

    if 'PreFormAnswers' in full_df.columns:

        full_df['PreFormAnswers'] = (
            full_df['PreFormAnswers']
            .apply(
                lambda x:
                    f"A={x[0]},C={x[1]},I={x[2]},M={x[3]}"
                    if isinstance(x, list)
                    else str(x)
            )
        )

    # ----------------------------------
    # montar dataframe colaborativo
    # ----------------------------------

    collab_df = build_collab_df(
        full_df,
        method
    )

    print("Interações:", len(collab_df))
    print("Usuários:", collab_df["User"].nunique())
    print("Itens:", collab_df["Item"].nunique())

    # =====================================================
    # LEAVE ONE OUT GLOBAL
    # =====================================================

    train_df, test_df = leave_one_out_split(
        collab_df,
        user_col='User'
    )

    print("Treino:", len(train_df))
    print("Teste:", len(test_df))

    # =====================================================
    # MATRIZ USER ITEM
    # =====================================================

    user_item_matrix = train_df.pivot_table(
        index='User',
        columns='Item',
        values='Rating',
        fill_value=0
    )

    # =====================================================
    # SIMILARIDADE
    # =====================================================

    if method % 2 == 0:

        item_similarity_df, user_similarity_df = (
            similiarity_pearson(user_item_matrix)
        )

    else:

        item_similarity_df, user_similarity_df = (
            similiarity_cosine(user_item_matrix)
        )

    # =====================================================
    # USER BASED
    # =====================================================

    rec_user_df = generate_recommendations(
        user_item_matrix,
        user_similarity_df,
        method="user"
    )

    # =====================================================
    # ITEM BASED
    # =====================================================

    rec_item_df = generate_recommendations(
        user_item_matrix,
        item_similarity_df,
        method="item"
    )

    # =====================================================
    # USER BASED METRICS
    # =====================================================

    hr_user = hit_rate_at_k(
        rec_user_df,
        test_df,
        k=top_k
    )

    mrr_user = mrr(
        rec_user_df,
        test_df
    )

    precision_user = precision_at_k(
        rec_user_df,
        test_df,
        k=top_k
    )

    recall_user = recall_at_k(
        rec_user_df,
        test_df,
        k=top_k
    )

    ndcg_user = ndcg_at_k(
        rec_user_df,
        test_df,
        k=top_k
    )

    # =====================================================
    # ITEM BASED METRICS
    # =====================================================

    hr_item = hit_rate_at_k(
        rec_item_df,
        test_df,
        k=top_k
    )

    mrr_item = mrr(
        rec_item_df,
        test_df
    )

    precision_item = precision_at_k(
        rec_item_df,
        test_df,
        k=top_k
    )

    recall_item = recall_at_k(
        rec_item_df,
        test_df,
        k=top_k
    )

    ndcg_item = ndcg_at_k(
        rec_item_df,
        test_df,
        k=top_k
    )

    # =====================================================
    # RESULTADOS
    # =====================================================

    results = {

        "HR_USER": hr_user,
        "MRR_USER": mrr_user,
        "PREC_USER": precision_user,
        "REC_USER": recall_user,
        "NDCG_USER": ndcg_user,

        "HR_ITEM": hr_item,
        "MRR_ITEM": mrr_item,
        "PREC_ITEM": precision_item,
        "REC_ITEM": recall_item,
        "NDCG_ITEM": ndcg_item,
    }

    print("\n======================================")
    print("RESULTADOS")
    print("======================================")

    print(results)

    return pd.DataFrame([results])

In [ ]:
# ==================== FUNÇÕES NOVAS/MODIFICADAS ====================

def item_stratified_split(df, item_col='Item', test_size=0.2, min_items_per_user=2):
    """
    Divide os dados garantindo que:
    1. Itens de teste existem no treino (mesmo item avaliado por outros usuários)
    2. Cada usuário tem pelo menos min_items_per_user itens no total
    """
    user_counts = df.groupby('User').size()
    valid_users = user_counts[user_counts >= min_items_per_user].index
    df = df[df['User'].isin(valid_users)].copy()

    item_counts = df.groupby(item_col).size()
    valid_items = item_counts[item_counts >= 2].index
    df = df[df[item_col].isin(valid_items)].copy()

    if len(df) == 0:
        print("⚠️ Sem dados suficientes após filtros!")
        return None, None

    unique_items = df[item_col].unique()
    train_items, test_items = train_test_split(
        unique_items,
        test_size=test_size,
        random_state=42
    )

    train_df = df[df[item_col].isin(train_items)].copy()
    test_df = df[df[item_col].isin(test_items)].copy()

    print(f"📊 Item-stratified split:")
    print(f"   Itens treino: {len(train_items)}")
    print(f"   Itens teste: {len(test_items)}")
    print(f"   Usuários treino: {train_df['User'].nunique()}")
    print(f"   Usuários teste: {test_df['User'].nunique()}")

    return train_df, test_df


def build_item_from_structure_discretized(df):
    """
    Versão DISCRETIZADA para estruturas granulares (métodos 9-12, 18-19)
    """
    df = df.copy()

    df['RoomBucket'] = pd.cut(
        df['TotalRooms'],
        bins=[0, 15, 25, 100],
        labels=['S', 'M', 'L']
    )

    df['KeyBucket'] = pd.cut(
        df['TotalKeys'],
        bins=[0, 3, 5, 10],
        labels=['F', 'N', 'M']
    )

    df['RevisitRatio'] = (df['RoomsEntered'] - df['UniqueRoomsEntered']) / df['TotalRooms']
    df['RevisitBucket'] = pd.cut(
        df['RevisitRatio'],
        bins=[0, 0.3, 0.7, 2],
        labels=['L', 'M', 'H']
    )

    return (
        "R" + df['RoomBucket'].astype(str) +
        "_K" + df['KeyBucket'].astype(str) +
        "_RR" + df['RevisitBucket'].astype(str)
    )


def collaborative_filtering_with_dfs(train_df, test_df, method):
    """
    Versão modificada que recebe train_df e test_df já prontos
    """
    print(f"\n{'='*60}")
    print(f"Executando Collaborative Filtering - Método {method}")
    print(f"{'='*60}")

    user_item_matrix = train_df.pivot_table(
        index='User',
        columns='Item',
        values='Rating',
        fill_value=0
    )

    print(f"Matriz de treino: {user_item_matrix.shape[0]} usuários x {user_item_matrix.shape[1]} itens")

    n_users = user_item_matrix.shape[0]
    n_items = user_item_matrix.shape[1]
    total_cells = n_users * n_items
    non_zero = (user_item_matrix > 0).sum().sum()
    sparsity = 1 - (non_zero / total_cells)
    print(f"Densidade: {100 * (1 - sparsity):.4f}%")

    if user_item_matrix.shape[1] < 2:
        print("❌ Poucos itens na matriz de treino!")
        return

    if method % 2 == 0:
        print("📊 Usando similaridade: Pearson")
        item_similarity_df, user_similarity_df = similiarity_pearson(user_item_matrix)
    else:
        print("📊 Usando similaridade: Cosseno")
        item_similarity_df, user_similarity_df = similiarity_cosine(user_item_matrix)

    predictions = []

    for _, row in test_df.iterrows():
        user = row['User']
        item = row['Item']
        true_rating = row['Rating']

        if item not in user_item_matrix.columns:
            continue
        if user not in user_item_matrix.index:
            continue

        if method in {1,3,5,7,9,11,14,16,18}:
            pred = predict_user_based(user, item, user_item_matrix, user_similarity_df)
        else:
            pred = predict_item_based(user, item, user_item_matrix, item_similarity_df)

        if pred > 0:
            predictions.append({
                'User': user,
                'Item': item,
                'True': true_rating,
                'Pred': pred
            })

    if len(predictions) == 0:
        print("❌ Nenhuma predição foi possível!")
        return

    pred_df = pd.DataFrame(predictions)

    rmse = np.sqrt(mean_squared_error(pred_df['True'], pred_df['Pred']))
    mae = mean_absolute_error(pred_df['True'], pred_df['Pred'])
    corr = pred_df['True'].corr(pred_df['Pred'])

    print(f"\n📈 Resultados para método {method}:")
    print(f"   Predições realizadas: {len(pred_df)}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE: {mae:.4f}")
    print(f"   Correlação: {corr:.4f}")

    return {
        'RMSE': rmse,
        'MAE': mae,
        'Predictions': len(pred_df),
        'Correlation': corr
    }


def classify_modified(_df, method):
    """
    Versão MODIFICADA da função classify com escolha de split baseada no método
    """
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) + ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]

    print("_df shape:", _df.shape)

    counts = _df.groupby("PlayerId").size()
    valid_users = counts[counts >= 2].index
    _df = _df[_df["PlayerId"].isin(valid_users)]

    # Métodos que precisam de item-stratified split
    if method in {1,2,3,4,9,10,11,12,14,15,16,17,18,19}:
        print(f"\n🎯 Usando Item-Stratified Split para método {method}")

        collab_df = build_collab_df(_df, method)
        train_collab, test_collab = item_stratified_split(collab_df, test_size=0.2)

        if train_collab is None or test_collab is None:
            print(f"❌ Método {method}: sem dados suficientes")
            return

        if method in {3,4,7,8,11,12}:
            test_users = set(test_collab['User'].unique())
            train_users = set(train_collab['User'].unique())
            valid_test_users = test_users & train_users
            test_collab = test_collab[test_collab['User'].isin(valid_test_users)]
            print(f"   Usuários comuns: {len(valid_test_users)}")

        collaborative_filtering_with_dfs(train_collab, test_collab, method)
        return

    else:
        # Métodos 5-8: Leave-One-Out Split
        print(f"\n🎯 Usando Leave-One-Out Split para método {method}")

        train_df, test_df = leave_one_out_split(_df)

        numeric_cols = train_df.select_dtypes(include=["float64", "int64"]).columns
        cols_to_exclude = ["PlayerID", "PreFormAnswers", "LevelName"]
        numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]

        scaler = MinMaxScaler()
        train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
        test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

        _dfmixed10 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
        _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
        _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
        _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
        _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

        collaborative_filtering(_dfmixed10, method, test_df)


# ==================== SUBSTITUIÇÃO NA FUNÇÃO build_collab_df ====================
# Na sua função build_collab_df, substitua a seção STRUCTURE_BASED por esta:

"""
# Dentro de build_collab_df, substitua:
elif method in STRUCTURE_ONLY_METHODS.union(STRUCTURE_PERF_METHODS):

    cols = ['PlayerId','TotalRooms','TotalKeys',
            'RoomsEntered','UniqueRoomsEntered']

    if method in STRUCTURE_ONLY_METHODS:
        cols.append('PostFormAnswers')
    else:
        cols += ['Success Rate','TotalLostHealth',
                 'EnemyKillRate','MapCompletion']

    df = _df[cols].copy()
    if method in STRUCTURE_ONLY_METHODS:
        df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'Rating']
    else:
        df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'SuccessRate',
                    'TotalLostHealth',
                    'EnemyKillRate',
                    'MapCompletion']

    # 🔥 USAR VERSÃO DISCRETIZADA
    df['Item'] = build_item_from_structure_discretized(df)

    if method in STRUCTURE_ONLY_METHODS:
        df['Rating'] = df['Rating'].apply(mean_post_forms)
    else:
        df['Rating'] = compute_performance_rating(df)

    df['Rating'] = minmax(df['Rating'])
    df = df[['User','Item','Rating']]
    return df[['User','Item','Rating']]
"""

"\n# Dentro de build_collab_df, substitua:\nelif method in STRUCTURE_ONLY_METHODS.union(STRUCTURE_PERF_METHODS):\n\n    cols = ['PlayerId','TotalRooms','TotalKeys',\n            'RoomsEntered','UniqueRoomsEntered']\n\n    if method in STRUCTURE_ONLY_METHODS:\n        cols.append('PostFormAnswers')\n    else:\n        cols += ['Success Rate','TotalLostHealth',\n                 'EnemyKillRate','MapCompletion']\n\n    df = _df[cols].copy()\n    if method in STRUCTURE_ONLY_METHODS:\n        df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'Rating']\n    else:\n        df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'SuccessRate',\n                    'TotalLostHealth',\n                    'EnemyKillRate',\n                    'MapCompletion']\n\n    # 🔥 USAR VERSÃO DISCRETIZADA\n    df['Item'] = build_item_from_structure_discretized(df)\n\n    if method in STRUCTURE_ONLY_METHODS:\n        df['Rating'] = df['Rati

In [ ]:
import random

import numpy as np
import json
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import os
import time
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split


from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# -----------------------------
# GRUPOS DE MÉTODOS
# -----------------------------
ITEM_PROFILE_METHODS = {1,2,5,6}
USER_PROFILE_METHODS = {3,4,7,8}
STRUCTURE_ONLY_METHODS = {9,10}
STRUCTURE_PERF_METHODS = {11,12,13}
HYBRID_PREFORM_METHODS = {14,15}
HYBRID_LEVEL_METHODS = {16,17}
HYBRID_STRUCTURE_METHODS = {18,19}
UNNAMED = {20,21}

# ------------------------------------------------- #
# 1- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 2- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 3- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 4- Entrada -> dungeons com perfis, ex: A=4, C=2, M=1, I=3; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 14 - Hibrido cosseno
# 15 - Hibrido pearson
# ------------------------------------------------- #

# 5- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 6- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *[OK]
# 7- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 8- Entrada -> nome das dungeons, ex: M:20T:30G:10E:40; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 16 - Hibrido cosseno
# 17 - Hibrido pearson
# ------------------------------------------------- #

# 9- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 10- Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada em item (nota -> PostFormAnswers)
# *
# 11 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Similiaridade cosseno; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *
# 12 - Entrada -> granular, ex: Dugeon com esses dados: 20 salas, 4 chaves-fechaduras, 1.5 de linearidad; Correlação de Pearson; Filtragem baseada no usuário (comportamento -> SuccessRate, TotalLostHealth_norm, EnemyKillRate, MapCompletion)
# *[OK]
# 18 - Hibrido cosseno
# 19 - Hibrido pearson
# ------------------------------------------------- #
# 13 - multiclasse
# ------------------------------------------------- #

class Results:
    accuracy = pd.DataFrame()
    precision = pd.DataFrame()
    recall = pd.DataFrame()
    f1 = pd.DataFrame()
    kappa = pd.DataFrame()
    train_accuracy = pd.DataFrame()
    train_precision = pd.DataFrame()
    train_recall = pd.DataFrame()
    train_f1 = pd.DataFrame()
    train_kappa = pd.DataFrame()
    cv_mean = pd.DataFrame()
    cv_stddev = pd.DataFrame()
    train_error_rate = pd.DataFrame()
    gap = pd.DataFrame()
    error_rate = pd.DataFrame()
    train_time = pd.DataFrame()
    infer_time = pd.DataFrame()

    accuracy["Setup"] = ""
    precision["Setup"] = ""
    recall["Setup"] = ""
    f1["Setup"] = ""
    cv_mean["Setup"] = ""
    cv_stddev["Setup"] = ""
    kappa["Setup"] = ""
    error_rate["Setup"] = ""
    train_time["Setup"] = ""
    infer_time["Setup"] = ""
    train_accuracy["Setup"] = ""
    train_precision["Setup"] = ""
    train_recall["Setup"] = ""
    train_f1["Setup"] = ""
    train_kappa["Setup"] = ""
    train_error_rate["Setup"] = ""
    gap["Setup"] = ""

    accuracy.set_index('Setup')
    precision.set_index('Setup')
    recall.set_index('Setup')
    f1.set_index('Setup')
    cv_mean.set_index('Setup')
    cv_stddev.set_index('Setup')
    kappa.set_index('Setup')
    error_rate.set_index('Setup')
    train_time.set_index('Setup')
    infer_time.set_index('Setup')
    train_accuracy.set_index('Setup')
    train_precision.set_index('Setup')
    train_recall.set_index('Setup')
    train_f1.set_index('Setup')
    train_kappa.set_index('Setup')
    train_error_rate.set_index('Setup')
    gap.set_index('Setup')

    def addModel(self, name):
        self.accuracy[name] = ""
        self.precision[name] = ""
        self.recall[name] = ""
        self.f1[name] = ""
        self.cv_mean[name] = ""
        self.cv_stddev[name] = ""
        self.kappa[name] = ""
        self.error_rate[name] = ""
        self.train_time[name] = ""
        self.infer_time[name] = ""
        self.train_accuracy[name] = ""
        self.train_precision[name] = ""
        self.train_recall[name] = ""
        self.train_f1[name] = ""
        self.train_kappa[name] = ""
        self.train_error_rate[name] = ""
        self.gap[name] = ""

    def addResults(self, _acc, _precision, _recall, _f1, _cv_mean, _cv_stddev, _kappa, _error_rate, _train_time, _infer_time, _train_acc, _train_precision, _train_recall, _train_f1, _train_kappa, _train_error_rate, _gap):
        self.accuracy.loc[len(self.accuracy)] = _acc
        self.precision.loc[len(self.precision)] = _precision
        self.recall.loc[len(self.recall)] = _recall
        self.f1.loc[len(self.f1)] = _f1
        self.cv_mean.loc[len(self.cv_mean)] = _cv_mean
        self.cv_stddev.loc[len(self.cv_stddev)] = _cv_stddev
        self.kappa.loc[len(self.kappa)] = _kappa
        self.error_rate.loc[len(self.error_rate)] = _error_rate
        self.train_time.loc[len(self.train_time)] = _train_time
        self.infer_time.loc[len(self.infer_time)] = _infer_time
        self.train_accuracy.loc[len(self.accuracy)] = _train_acc
        self.train_precision.loc[len(self.precision)] = _train_precision
        self.train_recall.loc[len(self.recall)] = _train_recall
        self.train_f1.loc[len(self.f1)] = _train_f1
        self.train_kappa.loc[len(self.kappa)] = _train_kappa
        self.train_error_rate.loc[len(self.error_rate)] = _train_error_rate
        self.gap.loc[len(self.gap)] = _gap


def initialize_data_frame():
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 2000)
    sns.set(rc={'figure.figsize':(19.20, 10.80)})


def get_dungeon_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['dungeons']).T


def get_user_data_frame():
    with open('backup.json') as data_file:
        json_data = json.load(data_file)
    return pd.DataFrame(json_data['__collections__']['users']).T


def drop_invalid_rows(_data_frame):
    _data_frame.drop(_data_frame[_data_frame['UniqueRoomsEntered'] == 0].index, inplace=True)
    return _data_frame


def drop_unused_columns(_data_frame):
    _data_frame.drop('__collections__', inplace=True, axis=1)
    _data_frame = _data_frame.drop(list(_data_frame.filter(regex='^Total.*Quests')), axis=1)
    _data_frame.drop('CompletedQuests', inplace=True, axis=1)
    _data_frame.drop('CompletedKillQuests', inplace=True, axis=1)
    _data_frame.rename(columns={'CompletedDamageQuests': 'CompletedKillQuests'}, inplace=True)
    _data_frame.drop('TreasuresCollected', inplace=True, axis=1)
    _data_frame.drop('TotalNpcs', inplace=True, axis=1)
    _data_frame.drop('WeaponName', inplace=True, axis=1)
    _data_frame.drop('HeatMap', inplace=True, axis=1)
    return _data_frame


def get_percentages(_data_frame):
    _data_frame['LockUsage'] = _data_frame['LocksOpened'].div(_data_frame['TotalLocks'].values)*100
    _data_frame['KeyCollection'] = _data_frame['KeysCollected'].div(_data_frame['TotalKeys'].values)*100
    _data_frame['MapCompletion'] = _data_frame['UniqueRoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['RoomRevisitRate'] = _data_frame['RoomsEntered'].div(_data_frame['TotalRooms'].values)*100
    _data_frame['EnemyKillRate'] = _data_frame['EnemiesKilled'].div(_data_frame['TotalEnemies'].values)*100
    _data_frame['TotalAttempts'] = _data_frame['TotalWins'] + _data_frame['TotalDeaths'].values
    _data_frame['TotalAttempts'] = _data_frame['TotalAttempts'].astype(float)
    _data_frame['TotalWins'] = _data_frame['TotalWins'].astype(float)
    _data_frame['Success Rate'] = _data_frame['TotalAttempts'].div(_data_frame['TotalWins'].values)*100
    _data_frame.replace([np.inf, -np.inf], 0, inplace=True)
    return _data_frame


def drop_processed_columns(_data_frame):
    _data_frame.drop('LocksOpened', inplace=True, axis=1)
    _data_frame.drop('TotalLocks', inplace=True, axis=1)
    _data_frame.drop('KeysCollected', inplace=True, axis=1)
    _data_frame.drop('TotalKeys', inplace=True, axis=1)
    _data_frame.drop('EnemiesKilled', inplace=True, axis=1)
    _data_frame.drop('TotalEnemies', inplace=True, axis=1)
    _data_frame.drop('TotalAttempts', inplace=True, axis=1)
    _data_frame.drop('TotalWins', inplace=True, axis=1)
    _data_frame.drop('TotalDeaths', inplace=True, axis=1)
    _data_frame.drop('NpcsInteracted', inplace=True, axis=1)
    _data_frame.drop('RoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalRooms', inplace=True, axis=1)
    _data_frame.drop('UniqueRoomsEntered', inplace=True, axis=1)
    _data_frame.drop('TotalTreasure', inplace=True, axis=1)
    return _data_frame


def split_preform_answers(_data_frame):
    _data_frame[['A', 'C', 'I', 'M']] = _data_frame['PreFormAnswers'].str.split(',', expand=True)
    _data_frame['A'] = _data_frame['A'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['C'] = _data_frame['C'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['I'] = _data_frame['I'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    _data_frame['M'] = _data_frame['M'].apply(lambda x: ''.join((char for char in x if char.isdigit())))
    return _data_frame


def augment_dataframe_with_scaling(df, y, n, scale_factor):
    """
    Augments the DataFrame by adding `n` new rows, scaling numeric columns by a given factor.
    The new rows inherit the same 'y' values as their corresponding sampled rows.

    Parameters:
        df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        n (int): Number of new rows to generate.
        scale_factor (float): The factor by which numeric columns will be scaled.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify numeric columns, excluding 'y'
    numeric_columns = df.select_dtypes(include=['number']).columns.difference(y)
    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = df.sample(n=n, replace=True).reset_index(drop=True)

    # Scale numeric columns
    for col in numeric_columns:
        sampled_rows[col] *= scale_factor

    # Append the scaled rows to the original DataFrame
    augmented_df = pd.concat([df, sampled_rows], ignore_index=True)

    return augmented_df


def augment_data_random(_df, y, n):
    """
    Augments the DataFrame by adding `n` new rows.
    - Adds noise to numerical columns using a normal distribution.
    - Retains the same 'y' values for the newly added rows as their corresponding sampled rows.

    Parameters:
        _df (pd.DataFrame): Original DataFrame with numeric columns and a 'y' column.
        y (list): the column(s) that will have their value retained
        n (int): Number of new rows to generate.

    Returns:
        pd.DataFrame: DataFrame with the original and new rows.
    """
    # Identify float and integer columns, excluding 'y'
    float_columns = _df.select_dtypes(include=['float']).columns.difference(y)
    int_columns = _df.select_dtypes(include=['integer']).columns.difference(y)

    # Randomly sample `n` rows from the original DataFrame
    sampled_rows = _df.sample(n=n, replace=True).reset_index(drop=True)

    # Add noise to float columns
    for col in float_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    for col in int_columns:
        if random.randint(1, 100) > 50:
            std = _df[col].std()
            noise = np.random.normal(loc=0, scale=std, size=n)
            sampled_rows[col] += noise

    # Ensure integer columns remain integers
    sampled_rows[int_columns] = sampled_rows[int_columns].round().astype(int)

    # Append the sampled (augmented) rows to the original DataFrame
    augmented_df = pd.concat([_df, sampled_rows], ignore_index=True)
    return augmented_df

def recommend_from_other_user(target_user, other_user, user_item_matrix):

    print(f"\nRecomendando para usuário {target_user} com base no usuário {other_user}")

    # Itens que target_user já jogou
    target_items = user_item_matrix.loc[target_user]

    # Itens que other_user já jogou
    other_items = user_item_matrix.loc[other_user]

    # Pegar apenas itens que:
    # - other_user jogou (rating > 0)
    # - target_user NÃO jogou (rating == 0)
    candidate_items = other_items[
        (other_items > 0) & (target_items == 0)
    ]

    if len(candidate_items) == 0:
        print("Nenhum item novo para recomendar.")
        return

    # Ordenar pelos melhores ratings do other_user
    recommendations = candidate_items.sort_values(ascending=False)

    print("\nItens recomendados:")
    print(recommendations.head(5))


# ---------------- RECOMENDAÇÃO USER-BASED EXPLICÁVEL ---------------- #

def recommend_user_based_explain(target_user, user_item_matrix, user_similarity_df, top_n=5):

    print(f"\nRecomendações para usuário {target_user}")

    # Similaridade do usuário alvo com todos os outros
    similarities = user_similarity_df.loc[target_user].drop(target_user)
    # Pegar itens que o usuário ainda NÃO jogou
    user_ratings = user_item_matrix.loc[target_user]
    items_not_played = user_ratings[user_ratings == 0].index

    predictions = {}
    explanations = {}

    for item in items_not_played:
        # Usuários que avaliaram esse item
        users_who_rated = user_item_matrix[user_item_matrix[item] > 0].index

        numerator = 0
        denominator = 0
        contributors = []

        for other_user in users_who_rated:

            sim = similarities.get(other_user, 0)

            if sim > 0:# só similaridade positiva
                rating = user_item_matrix.loc[other_user, item]
                contribution = sim * rating

                numerator += contribution
                denominator += abs(sim)

                contributors.append({
                    "user": other_user,
                    "similarity": sim,
                    "rating": rating,
                    "contribution": contribution
                })

        if denominator > 0:
            predicted_rating = numerator / denominator
            predictions[item] = predicted_rating
            explanations[item] = contributors

    if len(predictions) == 0:
        print("Nenhum item possível para recomendar.")
        return

    recommendations = (
        pd.Series(predictions)
        .sort_values(ascending=False)
        .head(top_n)
    )

    '''print("\nItens recomendados:")
    print(recommendations)'''

    # Mostrar explicação detalhada
    for item in recommendations.index:
        print("\n---------------------------------------")
        print(f"Explicação para {item}")
        print("---------------------------------------")

        contribs = sorted(
            explanations[item],
            key=lambda x: x["contribution"],
            reverse=True
        )

        for c in contribs[:5]:  # top 5 contribuidores
            print(
                f"Usuário {c['user']} | "
                f"similaridade={round(c['similarity'],3)} | "
                f"nota={round(c['rating'],2)} | "
                f"contribuição={round(c['contribution'],3)}"
            )
# Recommend items similar to 'Item1'
def recommend_similar_items(item, similarity_df, top_n=3):
  return similarity_df[item].sort_values(ascending=False)[1:top_n+1]

def print_user_ratings(user_item_matrix, user_similarity_df, user1_idx=0, user2_idx=1):
    # seleciona usuários
    u1 = user_similarity_df.index[user1_idx]
    u2 = user_similarity_df.index[user2_idx]

    print("Notas do usuário 1:")
    print(user_item_matrix.loc[u1].dropna())

    print("\nNotas do usuário 2:")
    print(user_item_matrix.loc[u2].dropna())

def analyze_user_activity(user_item_matrix):
    # Contar quantos níveis cada usuário jogou
    levels_per_user = user_item_matrix.notna().sum(axis=1)

    # Filtrar usuários que jogaram mais de 1 nível
    users_more_than_one = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(users_more_than_one.head())

    # Pegar o primeiro deles
    if len(users_more_than_one) > 0:
        u = users_more_than_one.index[0]

        print("\nUsuário selecionado:", u)
        print("\nQuantidade de níveis jogados:", levels_per_user.loc[u])

        print("\nNotas desse usuário:")
        print(user_item_matrix.loc[u].dropna())
    else:
        print("Nenhum usuário jogou mais de um nível.")

    # Distribuição geral
    print("\nDistribuição:")
    print(levels_per_user.value_counts().sort_index())

def build_quest_df(df):

    quest_cols = [col for col in df.columns if "Completed" in col and "Quests" in col]

    quest_df = df[['PlayerId'] + quest_cols].copy()

    # transformar em formato longo
    quest_df = quest_df.melt(
        id_vars='PlayerId',
        var_name='Item',
        value_name='Rating'
    )

    # remover zeros
    quest_df = quest_df[quest_df['Rating'] > 0]

    quest_df.columns = ['User', 'Item', 'Rating']

    return quest_df

def build_enemy_df(df):

    df = df.copy()

    # dificuldade simples (você pode melhorar depois)
    df['EnemyDifficulty'] = pd.cut(
        df['TotalEnemies'],
        bins=3,
        labels=['Easy', 'Medium', 'Hard']
    )

    enemy_df = df[['PlayerId', 'EnemyDifficulty', 'EnemiesKilled']].copy()

    enemy_df.columns = ['User', 'Item', 'Rating']

    return enemy_df

def build_struct_df(df):

    df = df.copy()

    df['Item'] = (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str) +
        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)
    )

    '''df['Rating'] = df['PostFormAnswers'].apply(
        lambda x: sum(x)/len(x) if isinstance(x, list) else x
    )'''
    df['Rating'] = df['PostFormAnswers'].apply(
        lambda x: sum(x)/len(x) if isinstance(x,list) else (0 if x is None else x))


    struct_df = df[['PlayerId', 'Item', 'Rating']]
    struct_df.columns = ['User', 'Item', 'Rating']

    return struct_df

def multi_view_recommendation(df, target_user):

    quest_df  = build_quest_df(df)
    enemy_df  = build_enemy_df(df)
    struct_df = build_struct_df(df)

    # gerar matrizes
    def build_matrix(d):
        return d.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)

    quest_matrix  = build_matrix(quest_df)
    enemy_matrix  = build_matrix(enemy_df)
    struct_matrix = build_matrix(struct_df)

    # similaridade
    def sim(m):
        return pd.DataFrame(
            cosine_similarity(m),
            index=m.index,
            columns=m.index
        )

    quest_sim  = sim(quest_matrix)
    enemy_sim  = sim(enemy_matrix)
    struct_sim = sim(struct_matrix)

    print("\n===== QUEST =====")
    recommend_user_based_explain(target_user, quest_matrix, quest_sim)

    print("\n===== ENEMY =====")
    recommend_user_based_explain(target_user, enemy_matrix, enemy_sim)

    print("\n===== STRUCT =====")
    recommend_user_based_explain(target_user, struct_matrix, struct_sim)


def predict_item_based(user, item, user_item_matrix, item_similarity_df, k=20):
    if item not in item_similarity_df.columns or user not in user_item_matrix.index:
        return 0  # ou np.nan
    similar_items = item_similarity_df[item]
    user_ratings = user_item_matrix.loc[user]
    # 👉 Filtrar itens não avaliados pelo usuário
    mask = user_ratings != 0
    similar_items = similar_items[mask]; user_ratings = user_ratings[mask]
    if similar_items.empty:
        return 0
    # (Opcional) usar apenas os top-K vizinhos
    similar_items = similar_items.sort_values(ascending=False).head(k)
    user_ratings = user_ratings.loc[similar_items.index]
    numerator = (similar_items * user_ratings).sum()
    denominator = similar_items.abs().sum()
    return numerator/denominator if denominator != 0 else 0

def predict_user_based(user, item, user_item_matrix, user_similarity_df, k=20):
    if user not in user_similarity_df.index or item not in user_item_matrix.columns:
        return 0
    similar_users = user_similarity_df.loc[user].drop(user)
    item_ratings = user_item_matrix[item]
    mask = item_ratings != 0
    similar_users = similar_users[mask]; item_ratings = item_ratings[mask]
    if similar_users.empty:
        return 0
    similar_users = similar_users.sort_values(ascending=False).head(k)
    item_ratings = item_ratings.loc[similar_users.index]
    numerator = (similar_users * item_ratings).sum()
    denominator = similar_users.abs().sum()
    return numerator/denominator if denominator != 0 else 0


def predict_hybrid(user, item,
                   user_item_matrix,
                   item_similarity_df,
                   user_similarity_df,
                   alpha=0.5):

    score_item = predict_item_based(
        user, item,
        user_item_matrix,
        item_similarity_df
    )

    score_user = predict_user_based(
        user, item,
        user_item_matrix,
        user_similarity_df
    )

    return alpha * score_item + (1 - alpha) * score_user

def recommend_hybrid(user,
                     user_item_matrix,
                     item_similarity_df,
                     user_similarity_df,
                     top_n=5,
                     alpha=0.5):

    items = user_item_matrix.columns
    predictions = {}

    for item in items:
        if user_item_matrix.loc[user, item] == 0:  # não avaliou
            predictions[item] = predict_hybrid(
                user,
                item,
                user_item_matrix,
                item_similarity_df,
                user_similarity_df,
                alpha
            )

    ranked = sorted(predictions.items(),
                    key=lambda x: x[1],
                    reverse=True)

    return ranked[:top_n]

def mean_post_forms(x):
    return sum(x)/len(x) if isinstance(x, list) else x

def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def compute_performance_rating(df):
    df['TotalLostHealth_norm'] = minmax(df['TotalLostHealth'])

    return (
        df['SuccessRate']
        - df['TotalLostHealth_norm']
        + df['EnemyKillRate']
        + df['MapCompletion']
    )

def build_item_from_structure(df):
    df['TotalRooms'] = df['TotalRooms'].replace(0, 1)

    return (
        "R" + df['TotalRooms'].astype(str) +
        "_K" + df['TotalKeys'].astype(str) +
        "_RR" + (df['RoomsEntered'] - df['UniqueRoomsEntered']).astype(str)
    )

def compute_hybrid_rating(df, alpha=0.5):
    df_valid = df['PostFormAnswers'].apply(mean_post_forms)
    df_comp = compute_performance_rating(df)

    return alpha * df_comp + (1 - alpha) * df_valid

def build_collab_df(_df, method):

    # ----------------------------
    # ITEM PROFILE
    # ----------------------------
    if method in ITEM_PROFILE_METHODS:

        if method in {1,2}:
            item_col = 'PreFormAnswers'
        else:
            item_col = 'LevelName'

        cols = ['PlayerId', item_col, 'PostFormAnswers']

        df = _df[cols].copy()
        df.columns = ['User', 'Item', 'Rating']
        df['Rating'] = df['Rating'].apply(mean_post_forms)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]

    # ----------------------------
    # USER PROFILE (performance)
    # ----------------------------
    elif method in USER_PROFILE_METHODS:

        if method in {3,4}:
            item_col = 'PreFormAnswers'
        else:
            item_col = 'LevelName'

        cols = ['PlayerId', item_col, 'Success Rate',
                'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User', 'Item', 'SuccessRate',
                      'TotalLostHealth', 'EnemyKillRate', 'MapCompletion']


        df['Rating'] = compute_performance_rating(df)
        df = df[['User','Item','Rating']]
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]

    # ----------------------------
    # STRUCTURE BASED
    # ----------------------------
    elif method in STRUCTURE_ONLY_METHODS.union(STRUCTURE_PERF_METHODS):

        cols = ['PlayerId','TotalRooms','TotalKeys',
                'RoomsEntered','UniqueRoomsEntered']

        if method in STRUCTURE_ONLY_METHODS:
            cols.append('PostFormAnswers')
        else:
            cols += ['Success Rate','TotalLostHealth',
                     'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        if method in STRUCTURE_ONLY_METHODS:
          df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'Rating']
        else:
            df.columns = ['User', 'TotalRooms', 'TotalKeys', 'RoomsEntered', 'UniqueRoomsEntered', 'SuccessRate',
                        'TotalLostHealth',
                        'EnemyKillRate',
                        'MapCompletion']

        df['Item'] = build_item_from_structure(df)

        if method in STRUCTURE_ONLY_METHODS:
            df['Rating'] = df['Rating'].apply(mean_post_forms)
        else:

            df['Rating'] = compute_performance_rating(df)

        df['Rating'] = minmax(df['Rating'])
        df = df[['User','Item','Rating']]
        return df[['User','Item','Rating']]

    # --------------------------------------------------
    # HYBRID PREFORM (14,15)
    # --------------------------------------------------
    elif method in HYBRID_PREFORM_METHODS:

        cols = ['PlayerId','PreFormAnswers','PostFormAnswers',
                'Success Rate','TotalLostHealth',
                'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','Item','PostFormAnswers',
                      'SuccessRate','TotalLostHealth',
                      'EnemyKillRate','MapCompletion']

        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]


    # --------------------------------------------------
    # HYBRID LEVEL (16,17)
    # --------------------------------------------------
    elif method in HYBRID_LEVEL_METHODS:

        cols = ['PlayerId','LevelName','PostFormAnswers',
                'Success Rate','TotalLostHealth',
                'EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','Item','PostFormAnswers',
                      'SuccessRate','TotalLostHealth',
                      'EnemyKillRate','MapCompletion']

        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]


    # --------------------------------------------------
    # HYBRID STRUCTURE (18,19)
    # --------------------------------------------------
    elif method in HYBRID_STRUCTURE_METHODS:

        cols = ['PlayerId','TotalRooms','TotalKeys',
                'RoomsEntered','UniqueRoomsEntered',
                'PostFormAnswers','Success Rate',
                'TotalLostHealth','EnemyKillRate','MapCompletion']

        df = _df[cols].copy()
        df.columns = ['User','TotalRooms','TotalKeys',
                      'RoomsEntered','UniqueRoomsEntered',
                      'PostFormAnswers','SuccessRate',
                      'TotalLostHealth','EnemyKillRate','MapCompletion']

        df['Item'] = build_item_from_structure(df)
        df['Rating'] = compute_hybrid_rating(df)
        df['Rating'] = minmax(df['Rating'])
        return df[['User','Item','Rating']]
    elif method in UNNAMED:

      cols = [
          'PlayerId',

          # combate
          'EnemiesKilled',
          'MaxCombo',

          # quests combate
          'CompletedKillQuests',
          'CompletedDamageQuests'
      ]

      if method == 20:
            cols.append('PostFormAnswers')
      else:
            cols += ['Success Rate','TotalLostHealth','MapCompletion','EnemyKillRate']


      df = _df[cols].copy()

      # -----------------------------------
      # RENOMEAR
      # -----------------------------------

      if method == 20:

        df.columns = [
            'User',
            'EnemiesKilled',
            'MaxCombo',
            'CompletedKillQuests',
            'CompletedDamageQuests',
            'Rating'
        ]

      else:

          df.columns = [
              'User',
              'EnemiesKilled',
              'MaxCombo',
              'CompletedKillQuests',
              'CompletedDamageQuests',
              'SuccessRate',
              'TotalLostHealth',
              'MapCompletion',
              'EnemyKillRate',
          ]

      # ===================================
      # COMBAT SCORE
      # ===================================

      df['CombatScore'] = (
          df['EnemiesKilled'] +
          df['MaxCombo'] +
          df['CompletedKillQuests'] +
          df['CompletedDamageQuests']
      ) / 4

      # ===================================
      # NORMALIZA
      # ===================================

      df['CombatScore'] = minmax(          df['CombatScore']      )

      # ===================================
      # BUCKET
      # ===================================

      def combat_bucket(x):

          if x < 0.33:
              return 'COMBAT_LOW'

          elif x < 0.66:
              return 'COMBAT_MEDIUM'

          return 'COMBAT_HIGH'

      # ===================================
      # ITEM
      # ===================================

      df['Item'] = df['CombatScore'].apply(
          combat_bucket
      )

      # ===================================
      # RATING
      # ===================================

      if method == 20:
            df['Rating'] = df['Rating'].apply(mean_post_forms)
      else:
            df['Rating'] = compute_performance_rating(df)


      # ===================================
      # NORMALIZA RATING
      # ===================================

      df['Rating'] = minmax(
          df['Rating']
      )

      # ===================================
      # FINAL
      # ===================================

      df = df[['User', 'Item', 'Rating']]
      return df

    else:
        raise ValueError(f"Método {method} inválido.")

    return df

def analyze_user_item_matrix(user_item_matrix):

    print("------------- VERIFICAÇÃO DOS DADOS -------------")
    print("Número de usuários:", user_item_matrix.shape[0])
    print("Número de itens:", user_item_matrix.shape[1])

    print("\nQuantidade média de avaliações por usuário:")
    print(user_item_matrix.count(axis=1).mean())

    print("\nQuantidade média de usuários por item:")
    print(user_item_matrix.count(axis=0).mean())

    print("\n------------- VARIÂNCIA ITEM POR ITEM -------------")
    print("Variância de cada item:")
    print(user_item_matrix.var())

def similiarity_pearson(user_item_matrix):
    # Pearson item-item
    item_similarity_pearson = user_item_matrix.corr(method='pearson')

    print("\nITEM POR ITEM (PEARSON):\n")
    print(item_similarity_pearson)
    item_similarity_df = item_similarity_pearson

    user_similarity_pearson = user_item_matrix.T.corr(method='pearson')
    user_similarity_pearson = user_similarity_pearson.fillna(0)

    print("\nUSUARIO POR USUARIO (PEARSON):\n")
    print(user_similarity_pearson)

    user_similarity_df  = user_similarity_pearson

    return item_similarity_df, user_similarity_df

def similiarity_cosine(user_item_matrix):
    # Compute item similarity using cosine similarity
    item_similarity = cosine_similarity(user_item_matrix.T)
    item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

    #centralizando
    user_item_centered = user_item_matrix.sub(user_item_matrix.mean(axis=1), axis=0)
    #print("\nconsseno cetnralizado:\n")
    #print(user_item_centered)

    # Similaridade entre itens
    # item por item
    item_similarity_centered = cosine_similarity(user_item_centered.T)
    item_similarity_centered_df = pd.DataFrame(item_similarity_centered, index=user_item_centered.columns, columns=user_item_centered.columns)

    #print("\nconsseno:\n")
    #print(item_similarity_df)

    #print("\nconsseno cetnralizado:\n")
    #print(item_similarity_centered)

    # usuario por usuário
    user_similarity = cosine_similarity(user_item_centered)
    user_similarity_df = pd.DataFrame( user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index )
    #print("\nUSUARIO POR USUARIO:\n")
    #print(user_similarity_df.iloc[:50, :50])

    return item_similarity_df, user_similarity_df

def train_test_split_interactions(df, test_size=0.2):
    train, test = train_test_split(df, test_size=test_size, random_state=42)
    return train, test

def predict_all_users(user_item_matrix, similarity_matrix, method="user"):
    predictions = []

    for user in user_item_matrix.index:
        for item in user_item_matrix.columns:
            if user_item_matrix.loc[user, item] == 0:

                if method == "user":
                    pred = predict_user_based(user, item, user_item_matrix, similarity_matrix)
                else:
                    pred = predict_item_based(user, item, user_item_matrix, similarity_matrix)

                predictions.append((user, item, pred))

    return pd.DataFrame(predictions, columns=["User", "Item", "Prediction"])

def evaluate_rating_metrics(predictions_df, test_matrix):

    y_true = []
    y_pred = []

    for _, row in predictions_df.iterrows():
        user = row["User"]
        item = row["Item"]

        if user in test_matrix.index and item in test_matrix.columns:
            true_rating = test_matrix.loc[user, item]
            if true_rating != 0:
                y_true.append(true_rating)
                y_pred.append(row["Prediction"])

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    return rmse, mae

def compute_rmse(test_df, prediction_col, rating_col):
    """
    Calcula RMSE ignorando:
        - valores NaN
        - casos sem predição
        - divisão por zero
    """

    # Manter apenas linhas com valor real e predito
    mask = (
        test_df[rating_col].notna() &
        test_df[prediction_col].notna()
    )

    y_true = test_df.loc[mask, rating_col].values
    y_pred = test_df.loc[mask, prediction_col].values

    if len(y_true) == 0:
        raise ValueError("Nenhuma predição válida para calcular RMSE.")

    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    return rmse

def precision_at_k(rec_df, test_df, k=5, threshold=1.0):
    precision_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        precision_list.append(hits / k)

    return np.mean(precision_list)
def recall_at_k(rec_df, test_df, k=5, threshold=1.0):
    recall_list = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[(test_df["User"] == user) & (test_df["Rating"] >= threshold)]["Item"].values

        if len(true_items) == 0:
            continue

        hits = len(set(top_k) & set(true_items))
        recall_list.append(hits / len(true_items))

    return np.mean(recall_list)
def hit_rate_at_k(rec_df, test_df, k=5):
    hits = 0
    users = test_df["User"].unique()

    for user in users:
        top_k = rec_df[rec_df["User"] == user].head(k)["Item"].values
        true_items = test_df[test_df["User"] == user]["Item"].values

        if len(set(top_k) & set(true_items)) > 0:
            hits += 1

    return hits / len(users)
def mrr(rec_df, test_df):
    rr_total = []

    for user in test_df["User"].unique():

        top_k = rec_df[rec_df["User"] == user].reset_index()

        true_items = set(test_df[test_df["User"] == user]["Item"])

        rank = 0

        for i, row in top_k.iterrows():
            if row["Item"] in true_items:
                rank = i + 1
                break

        if rank > 0:
            rr_total.append(1 / rank)

    return np.mean(rr_total)



def collaborative_filtering(_df, method, teste):

    # ------------- VISUALIZAÇÃO DOS DADOS ------------
    print("--------------------1----------------------------")
    print(_df.shape)
    print(_df.head())
    print("-"*80)

    print("\nFrequência no dungeon_frame final:")
    print(_df['PreFormAnswers'].value_counts())

    # Número de usuários únicos pelo PlayerId
    num_unicos = _df['PlayerId'].nunique()
    print(f"\nNúmero de usuários únicos: {num_unicos}")

    # ------------- NORMALIZAÇÃO DOS DADOS ------------#
    print("--------------------2----------------------------")
    collab_df = build_collab_df(_df, method)
    test_df = build_collab_df(teste, method)

    print("\nMatriz para filtragem 10 :")
    print(test_df.head())

    print("\nMatriz para filtragem 1 :")
    print(collab_df.head())

    # ------------- CÁLCULOS ------------#
    print("--------------------3----------------------------")

    # Create user-item matrix
    user_item_matrix = collab_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
    test_matrix = test_df.pivot_table(index='User', columns='Item', values='Rating', fill_value=0)
    print(test_matrix.head())

    # 🔥 COLOCA AQUI ↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓↓

    n_users = user_item_matrix.shape[0]
    n_items = user_item_matrix.shape[1]

    total_cells = n_users * n_items
    non_zero = (user_item_matrix > 0).sum().sum()

    sparsity = 1 - (non_zero / total_cells)

    print("\n===== SPARSITY =====")
    print("Users:", n_users)
    print("Items:", n_items)
    print("Total cells:", total_cells)
    print("Non-zero interactions:", non_zero)
    print("Sparsity:", sparsity)

    # ------------- VERIFICAÇÃO DOS DADOS ------------#
    print("--------------------4----------------------------")
    analyze_user_item_matrix(user_item_matrix)

    # ---------------- SIMILIARIDADE
    print("--------------------5----------------------------")
    if(method % 2 == 0):
      item_similarity_df, user_similarity_df = similiarity_pearson(user_item_matrix)
    else:
      item_similarity_df, user_similarity_df = similiarity_cosine(user_item_matrix)

    # ------------- RECOMENDAR ITEM POR ITEM ------------#
    # esta comentado, pq nao fareirpos oturos casos
    '''if(method >=1 and method <=4):
      # Example recommendation
      similar_items = recommend_similar_items('A=4, C=3, I=2, M=1', item_similarity_df)
      print("\nItems similar to A=4, C=3, I=2, M=1:", similar_items)'''

    # Recomendar todos os itens por similarida itens
    print("--------------------6----------------------------")
    all_recommendations = {}

    for item in user_item_matrix.columns:
        all_recommendations[item] = recommend_similar_items(item, item_similarity_df)

    print(all_recommendations)

    # ------------- SIMILIARIDA USUARIO POR USUARIO ------------#
    print("--------------------7----------------------------")

    # print abaixo é pra ENTENDNEO O RESULTADO ACIMA
    print_user_ratings(user_item_matrix, user_similarity_df)

    # ------------- testando um usuario com mais de um nivel ------------#
    print("--------------------8----------------------------")
    analyze_user_activity(user_item_matrix)

    # Recomendar para u1 baseado em u2
    print("--------------------9----------------------------")
    recommend_from_other_user(user_item_matrix.index[0], user_item_matrix.index[1], user_item_matrix)

    # Recomendar para u2 baseado em u1
    print("--------------------10----------------------------")
    recommend_from_other_user(user_item_matrix.index[1], user_item_matrix.index[0], user_item_matrix)

    # Escolher usuário
    print("--------------------13----------------------------")
    recommend_user_based_explain(user_item_matrix.index[0], user_item_matrix, user_similarity_df, top_n=5)

    # 🔥 MULTI-VIEW RECOMMENDATION (NOVO)
    print("--------------------14----------------------------")
    if method == 13:
        print("\n\n=========== MULTI VIEW ===========")
        multi_view_recommendation(_df, _df['PlayerId'].iloc[0])
    elif (method >= 14):
        print("\n=========== HYBRID RECOMMENDATION ===========")
        recommend_hybrid(user_item_matrix.index[0], user_item_matrix, item_similarity_df, user_similarity_df)


    # ---------------------------------------------
    # PREDIÇÃO APENAS NO TEST SET
    # ---------------------------------------------

    # 1. Garantir interseção estrutural
    print("TEST MATRIX SHAPE BEFORE:", test_matrix.shape)
    print("TRAIN MATRIX SHAPE:", user_item_matrix.shape)

    common_users = set(collab_df["User"]).intersection(test_df["User"])
    common_items = set(collab_df["Item"]).intersection(test_df["Item"])

    print("\nANTES DO FILTRO:")
    print("Test size:", len(test_df))
    print("Common users:", len(common_users))
    print("Common items:", len(common_items))

    test_df = test_df[
        test_df["User"].isin(common_users) &
        test_df["Item"].isin(common_items)
    ]

    print("\nDEPOIS DO FILTRO:")
    print("Test size:", len(test_df))

    if len(test_df) == 0:
        print("\n TEST DF VIROU VAZIO → pulando métricas")
        return


    print("TEST MATRIX SHAPE AFTER:", test_matrix.shape)


    all_items = user_item_matrix.columns

    recommendations_user = []

    for user in test_df["User"].unique():

        for item in all_items:

            score = predict_user_based(
                user,
                item,
                user_item_matrix,
                user_similarity_df
            )

            recommendations_user.append({
                "User": user,
                "Item": item,
                "Score": score
            })
    recommendations_item = []

    for user in test_df["User"].unique():

        for item in all_items:

            score = predict_item_based(
                user,
                item,
                user_item_matrix,
                item_similarity_df
            )

            recommendations_item.append({
                "User": user,
                "Item": item,
                "Score": score
            })
    rec_user_df = pd.DataFrame(recommendations_user)

    rec_user_df = rec_user_df.sort_values(
        ["User", "Score"],
        ascending=[True, False]
    )

    rec_item_df = pd.DataFrame(recommendations_item)

    rec_item_df = rec_item_df.sort_values(
        ["User", "Score"],
        ascending=[True, False]
    )

    predictions = []
    predictions_item = []

    for _, row in test_df.iterrows():

        user = row["User"]
        item = row["Item"]
        true_rating = row["Rating"]

        # ---------- USER-BASED ----------
        pred_user = predict_user_based(
            user,
            item,
            user_item_matrix,
            user_similarity_df
        )

        # ---------- ITEM-BASED ----------
        pred_item = predict_item_based(
            user,
            item,
            user_item_matrix,
            item_similarity_df
        )

        predictions.append({
            "User": user,
            "Item": item,
            "Rating": true_rating,
            "Prediction": pred_user
        })

        predictions_item.append({
            "User": user,
            "Item": item,
            "Rating": true_rating,
            "Prediction": pred_item
        })


    print("\n===== ITEM-BASED =====")

    pred_df = pd.DataFrame(predictions)
    pred_item_df = pd.DataFrame(predictions_item)

    print("\n===== DEBUG INTERSEÇÃO =====")
    print("Itens únicos preditos:", pred_df['Item'].nunique())
    print("Itens únicos no teste:", test_df['Item'].nunique())

    intersection = set(pred_df['Item']) & set(test_df['Item'])
    print("Interseção de itens:", len(intersection))
    print("Itens em comum:", intersection)

    print("\n===== DEBUG USUÁRIOS =====")
    print(test_df.groupby("User")["Item"].count())

    # remover NaN
    pred_df = pred_df.dropna()
    pred_item_df = pred_item_df.dropna()

    rmse = np.sqrt(mean_squared_error(pred_df["Rating"], pred_df["Prediction"]))
    mae  = mean_absolute_error(pred_df["Rating"], pred_df["Prediction"])
    print("Predições válidas:", len(pred_df))
    print("RMSE:", rmse)
    print("MAE:", mae)
    rmse_item = np.sqrt(mean_squared_error(pred_item_df["Rating"], pred_item_df["Prediction"]))
    mae_item  = mean_absolute_error(pred_item_df["Rating"], pred_item_df["Prediction"])
    print("RMSE:", rmse_item)
    print("MAE:", mae_item)
    precision = precision_at_k(pred_df, test_df, k=5, threshold=1.0)
    recall = recall_at_k(pred_df, test_df, k=5, threshold=1.0)

    precision_user = precision_at_k(rec_user_df, test_df, k=8, threshold=0.0)
    recall_user    = recall_at_k(rec_user_df, test_df, k=8, threshold=0.0)
    precision_item = precision_at_k(rec_item_df, test_df, k=8, threshold=0.0)
    recall_item    = recall_at_k(rec_item_df, test_df, k=8, threshold=0.0)

    print("Precision@5:", precision_user)
    print("Recall@5:", recall_user)
    print("Precision@5:", precision_item)
    print("Recall@5:", recall_item)

    hr_user = hit_rate_at_k(rec_user_df, test_df, k=8)
    mrr_user = mrr(rec_user_df, test_df)

    hr_item = hit_rate_at_k(rec_item_df, test_df, k=8)
    mrr_item = mrr(rec_item_df, test_df)

    print("HitRate@5:", hr_user)
    print("MRR:", mrr_user)
    print("HitRate@5:", hr_item)
    print("MRR:", mrr_item)


    return {
        'RMSE': rmse,
        'MAE': mae,
        #'Precision@5': precision,
        #'Recall@5': recall,
        'RMSE_item': rmse_item,
        'MAE_item': mae_item,
        #'Precision@5_item': precision_item,
        #'Recall@5_item': recall_item
    }


def leave_one_out_split(df, user_col='PlayerId'):
    train_list, test_list = [], []
    for u, grp in df.groupby(user_col):
        if len(grp)<2:
            train_list.append(grp); continue
        test_list.append(grp.sample(1, random_state=42))
        train_list.append(grp.drop(test_list[-1].index))
    return pd.concat(train_list), pd.concat(test_list)


def classify(_df, method):
    _df['PreFormAnswers'] = _df['PreFormAnswers'].apply(lambda x: 'A=' + str(x[0]) + ", C=" + str(x[1]) + ", I=" + str(x[2]) +
                                                                  ", M=" + str(x[3]))
    _df = split_preform_answers(_df)
    n = _df.shape[0]

    print("_df shape:", _df.shape)

    # 🔴 FILTRO ANTES DO SPLIT (ESSENCIAL)
    counts = _df.groupby("PlayerId").size()
    valid_users = counts[counts >= 2].index
    _df = _df[_df["PlayerId"].isin(valid_users)]

    # 🔵 SPLIT
    train_df, test_df = leave_one_out_split(_df)


    numeric_cols = train_df.select_dtypes(include=["float64", "int64"]).columns

    # REMOVER colunas estruturais
    cols_to_exclude = ["PlayerID", "PreFormAnswers", "LevelName"]
    numeric_cols = [c for c in numeric_cols if c not in cols_to_exclude]

    scaler = MinMaxScaler()

    train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])



    _dfnoise = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfscale = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfnoise2 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 2*n)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)
    _dfscale2 = augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)
    _dfmixed = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n)
    _dfmixed = pd.concat([_dfmixed, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = augment_data_random(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], 6*n)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.05)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.95)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 1.10)], ignore_index=True)
    _dfmixed10 = pd.concat([_dfmixed10, augment_dataframe_with_scaling(train_df, ['PreFormAnswers', 'A', 'C', 'I', 'M'], n, 0.9)], ignore_index=True)

    dataframes = [#_df, _dfnoise, _dfscale, _dfnoise2, _dfscale2, _dfmixed,
                  _dfmixed10]
    experiment_names = [#'Baseline', '1xN', '1xS-1_05', '2xN', '2xS-1_05+0_95', '1xS-1_05+1xN',
                        '6xN+4xS-1_05+0_95+1_1+0_9']

    strategies = [#'Original', '1xNoise', '1xScale', '2xN', '2xScale', '1xN+1xS',
                  '6xN+4xS']

    # Quantos níveis diferentes cada usuário jogou
    levels_per_user = _df.groupby("PlayerId")["LevelName"].nunique()

    # Filtrar apenas quem jogou mais de 1 nível
    multi_level_users = levels_per_user[levels_per_user > 1]

    print("Usuários que jogaram mais de um nível:")
    print(multi_level_users)

    print("\nDetalhamento:\n")

    '''for user in multi_level_users.index:
        levels = _df[_df["PlayerId"] == user]["LevelName"].unique()
        print(f"Usuário {user} jogou:")
        print(list(levels))
        print("-" * 50)'''

    collaborative_filtering(_dfmixed10, method, test_df)



def reset_results():
    Results.accuracy = pd.DataFrame()
    Results.precision = pd.DataFrame()
    Results.recall = pd.DataFrame()
    Results.f1 = pd.DataFrame()
    Results.kappa = pd.DataFrame()
    Results.train_accuracy = pd.DataFrame()
    Results.train_precision = pd.DataFrame()
    Results.train_recall = pd.DataFrame()
    Results.train_f1 = pd.DataFrame()
    Results.train_kappa = pd.DataFrame()
    Results.cv_mean = pd.DataFrame()
    Results.cv_stddev = pd.DataFrame()
    Results.train_error_rate = pd.DataFrame()
    Results.gap = pd.DataFrame()
    Results.error_rate = pd.DataFrame()
    Results.train_time = pd.DataFrame()
    Results.infer_time = pd.DataFrame()

def main(method):
    initialize_data_frame()
    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()
    dungeon_frame = pd.merge(dungeon_frame, user_frame[['PlayerId', 'PreFormAnswers']], on=['PlayerId'], how='left')
    dungeon_frame = drop_invalid_rows(dungeon_frame)
    #dungeon_frame = drop_unused_columns(dungeon_frame)
    dungeon_frame = get_percentages(dungeon_frame)
    #dungeon_frame = drop_processed_columns(dungeon_frame)
    classify(dungeon_frame, method)
    '''evaluate_collaborative_filtering(
        dungeon_frame,
        method=1,
        top_k=10
    )'''


def teste():
    initialize_data_frame()

    dungeon_frame = get_dungeon_data_frame()
    user_frame = get_user_data_frame()

    # 🔥 CONVERTE PARA TUPLA AQUI
    profiles = user_frame['PreFormAnswers'].apply(tuple)

    print("Combinações únicas:")
    print(sorted(profiles.unique()))

    print("\nQuantidade de combinações únicas:")
    print(profiles.nunique())

    print("\nFrequência de cada combinação:")
    print(profiles.value_counts())

    dungeon_frame = pd.merge(
        dungeon_frame,
        user_frame[['PlayerId', 'PreFormAnswers']],
        on=['PlayerId'],
        how='left'
    )

    print("\nFrequência no dungeon_frame final:")
    print(dungeon_frame['PreFormAnswers'].value_counts())

    # Listar todas as colunas
    colunas = dungeon_frame.columns.tolist()

    print(f"\nTotal de colunas: {len(colunas)}\n")

    # Exibir em ordem alfabética numerada
    for i, col in enumerate(sorted(colunas), 1):
        print(f"{i:3d}. {col}")

In [ ]:
main(5) # 1 E 5 FOI, VER PQ 9 NAO FOI

/tmp/ipykernel_1945/3683710570.py:220: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  _data_frame.replace([np.inf, -np.inf], 0, inplace=True)


_df shape: (119, 67)
Usuários que jogaram mais de um nível:
PlayerId
53627948       4
78194049       3
81972292      11
121412094     12
282268519      4
1093542367    12
1153731241     4
1776427228     2
2079201442     4
2079942857    12
2096844718     3
Name: LevelName, dtype: int64

Detalhamento:

--------------------1----------------------------
(1710, 67)
                      PostFormAnswers  TotalEnemies  TotalMasteryQuests  MaxCombo  TotalExploreQuests  TotalNpcs  CompletedReportQuests  CompletedListenQuests  TotalDeaths  CompletedAchievementQuests  TotalGiveQuests  TotalAchievementQuests  TotalKeys  PlayerId  TotalLocks  KeysCollected  TotalExchangeQuests  TotalLostHealth  CompletedExploreQuests  CompletedExchangeQuests  TotalGoToQuests  RoomsEntered  CompletedDamageQuests  TotalAttempts  TotalCreativityQuests  TotalGatherQuests  NpcsInteracted  CompletedMasteryQuests                                       InputProfile  CompletedGiveQuests WeaponName  TotalImmersionQuests      

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


#TODO
- 1º Analises -> precsiao,recall, rmse, mae
- 2º Fazer a tabela para cada atribulo - inmigos, quest, difuclda, prefenais (maar, explora...) (deixre indicado para cada um) um multiclasse
- 3º Normalizar os dados (1 a 0)
- 4º PCA (usar todos os dados e depois usar o pca para ver quais sao inuties, assim tirar eles)
- 5º fazer as analises como dito no artigo (eu tenho que faze-las
- 6º Organizar o código mais ainda

#add no artigo
- poderia usar o perfil para melhora a recomendação
- recomendaco a aprtir de todos usuarios? ex.: um grupo aprova isso e vc ta no grupo, entao vc vai gostar



